# ⚠️ Limitations and Honest Assessment

This notebook quantifies the known limitations of the diff-integrator approach to NMR structure refinement. Scientific credibility requires honest disclosure of where the method succeeds and where it does not.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins-lab/diff-integrator/blob/main/examples/interactive_tutorials/limitations.ipynb)

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/elkins-lab/diff-integrator.git
    %pip install -q ./diff-integrator matplotlib seaborn numpy pandas
    import os
    os.chdir('diff-integrator/examples/interactive_tutorials')

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
RES = Path("../../benchmarks/results")
COLORS = {"2KZV": "#2ecc71", "GmR58A": "#3498db", "HR2876B": "#e74c3c"}

## Limitation 1: NeRF Geometric Drift

diff-integrator uses a **NeRF (Natural Extension Reference Frame)** builder to convert backbone dihedral angles (φ, ψ) into Cartesian coordinates. The NeRF builder assumes **ideal bond lengths and angles** for every residue:
- N–Cα: 1.46 Å
- Cα–C: 1.52 Å  
- Bond angle at Cα: 111°

Real PDB structures have small deviations from these ideal values. These deviations **accumulate** along the chain: each residue's deviation from ideal geometry adds a small positional error that compounds over the chain.

### Impact
The Cα RMSD between the NeRF-rebuilt starting structure and the raw PDB model 1 is measured as the **NeRF drift**. The optimizer sees this drifted structure as its starting point, not the true PDB coordinates.

In [ ]:
drifts = []
names = []
colors = []
for name, c in COLORS.items():
    try:
        d = float(np.load(RES / name / 'nerf_drift.npy'))
        drifts.append(d)
        names.append(name)
        colors.append(c)
    except FileNotFoundError:
        pass

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(names, drifts, color=colors)

ax.axvline(x=2.0, color='gray', linestyle='--', alpha=0.7)
ax.text(2.1, len(names)-0.5, 'Typical crystallographic resolution (2.0 Å)', color='gray')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.1, bar.get_y() + bar.get_height()/2, f'{width:.2f} Å',
            ha='left', va='center')

ax.set_xlabel('Kabsch RMSD (Å) between raw PDB Cα and NeRF-rebuilt Cα')
ax.set_title('NeRF Reconstruction Drift by Protein Length')
ax.set_xlim(0, max(drifts) + 2 if drifts else 10)
plt.tight_layout()
plt.show()

## Limitation 2: RDC Data-to-Parameter Ratio

The Saupe alignment tensor has **5 free parameters** ($D_a$, $R$, and 3 Euler angles describing the principal axis frame). Reliable tensor fitting requires many more RDC measurements than tensor parameters.

### The Cornilescu Recommendation
Bax & Tjandra recommend **at least 20 RDCs per alignment medium** for reliable tensor fitting. With fewer RDCs, the tensor can fit the data artificially without reflecting the true structural orientation.

### Benchmark Data-to-Parameter Ratios

| Protein | Medium | # RDCs | Tensor params | Ratio | Status |
|---|---|---|---|---|---|
| 2KZV | PAG | 23 | 5 | **4.6×** | Minimal ✓ |
| 2KZV | PEG | 16 | 5 | **3.2×** | Underdetermined ⚠️ |
| GmR58A | Gel+ | 43 | 5 | **8.6×** | Good ✓ |
| GmR58A | Gel− | 59 | 5 | **11.8×** | Excellent ✓ |
| GmR58A | PEG | 53 | 5 | **10.6×** | Excellent ✓ |
| HR2876B | PEG | 72 | 5 | **14.4×** | Excellent ✓ |
| HR2876B | Pf1 | 75 | 5 | **15.0×** | Gold standard ✓ |

Note: GmR58A and HR2876B RDC media are available in BMRB but have **not yet been incorporated** into diff-integrator. Only 2KZV currently uses RDC restraints.

In [ ]:
data = [
    ('2KZV PAG', 4.6),
    ('2KZV PEG', 3.2),
    ('GmR58A Gel+', 8.6),
    ('GmR58A Gel-', 11.8),
    ('GmR58A PEG', 10.6),
    ('HR2876B PEG', 14.4),
    ('HR2876B Pf1', 15.0)
]

labels = [d[0] for d in data]
ratios = [d[1] for d in data]

colors = ['#e74c3c' if r < 5 else '#f1c40f' if r < 10 else '#2ecc71' for r in ratios]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels, ratios, color=colors)

ax.axvline(x=5.0, color='red', linestyle='--', alpha=0.7)
ax.text(5.2, len(labels)-1, 'Minimum recommended (5.0)', color='red')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.2, bar.get_y() + bar.get_height()/2, f'{width:.1f}×',
            ha='left', va='center')

ax.set_xlabel('RDC Data-to-Parameter Ratio (higher is better)')
ax.set_title('RDC Overdetermination by Medium')
ax.set_xlim(0, 18)
plt.tight_layout()
plt.show()

## Limitation 3: RDC Overfitting on Underdetermined Datasets

The 2KZV PEG medium (16 RDCs, ratio 3.2×) demonstrates the overfitting problem clearly.

### What We Observe
- Q(PAG) before → after: **0.309 → 0.290** (modest, physically plausible improvement)
- Q(PEG) before → after: **0.373 → 0.047** (suspiciously large, below published NMR medoid)

The published NMR medoid Q-factor for 2KZV PEG is **0.36** (Li et al. 2023). The optimizer has driven Q(PEG) well below this value — not by finding a better structure, but by exploiting the underdetermined tensor fitting.

### Why This Happens
With only 16 RDCs, there are many distinct backbone conformations that simultaneously satisfy all 16 constraints while looking globally wrong. The optimizer finds one of these solutions and reports an artificially low Q-factor.

### The Safeguard in diff-integrator
The benchmark script explicitly prints a warning when Q(PEG) falls below 0.18 (50% below the NMR medoid), and the results README clearly labels the PEG result as likely overfitting. The **PAG Q-factor (0.290) is the reliable primary metric**.

In [ ]:
try:
    pag_before = float(np.load(RES / '2KZV/rdc_pag_q_before.npy'))
    pag_after = float(np.load(RES / '2KZV/rdc_pag_q_after.npy'))
    peg_before = float(np.load(RES / '2KZV/rdc_peg_q_before.npy'))
    peg_after = float(np.load(RES / '2KZV/rdc_peg_q_after.npy'))

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(2)
    width = 0.35

    ax.bar(x - width/2, [pag_before, peg_before], width, label='Initial', color='#95a5a6')
    ax.bar(x[0] + width/2, pag_after, width, label='Refined (PAG)', color='#2ecc71')
    ax.bar(x[1] + width/2, peg_after, width, label='Refined (PEG)', color='#e74c3c')

    # Ref lines
    ax.plot([-0.4, 0.4], [0.18, 0.18], 'k--', alpha=0.5)
    ax.text(-0.35, 0.19, 'Published Medoid (0.18)', fontsize=10)
    
    ax.plot([0.6, 1.4], [0.36, 0.36], 'k--', alpha=0.5)
    ax.text(0.65, 0.37, 'Published Medoid (0.36)', fontsize=10)

    ax.annotate('Genuine\nimprovement', xy=(0 + width/2, pag_after), xytext=(0 + width/2, pag_after + 0.1),
                arrowprops=dict(facecolor='#2ecc71', shrink=0.05), ha='center')
    ax.annotate('Likely overfitting', xy=(1 + width/2, peg_after), xytext=(1 + width/2, peg_after + 0.15),
                arrowprops=dict(facecolor='#e74c3c', shrink=0.05), ha='center')

    ax.set_xticks(x)
    ax.set_xticklabels(['PAG (23 RDCs)', 'PEG (16 RDCs)'])
    ax.set_ylabel('Pearson Q-Factor (lower is better)')
    ax.set_title('Demonstration of RDC Overfitting (2KZV)')
    ax.legend()
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("2KZV RDC data missing.")

## Limitation 4: Degrees-of-Freedom Imbalance

The optimizer has far more free parameters (backbone torsion angles) than experimental constraints per protein:

| Protein | Backbone DOFs (φ,ψ) | Cα shifts | RDC PAG | RDC PEG | Total constraints | DOF/constraint ratio |
|---|---|---|---|---|---|---|
| 2KZV | 2×92 = **184** | 91 | 23 | 16 | 130 | **1.4×** |
| GmR58A | 2×122 = **244** | 114 | — | — | 114 | **2.1×** |
| HR2876B | 2×107 = **214** | 97 | — | — | 97 | **2.2×** |

For GmR58A and HR2876B, there are **2× more free parameters than experimental constraints**. This means the optimizer can always find a torsion configuration that satisfies the data perfectly — but that solution is not unique and may not be physically meaningful.

This is why `GeometryLoss` is mandatory: it acts as a structural prior that eliminates the unconstrained directions in torsion space by penalizing deviation from the initial NMR structure.

### How Traditional Programs Solve This
X-PLOR, CNS, and CYANA address this by adding **NOE distance restraints** (often thousands per protein) and **Ramachandran dihedral priors**. These dramatically increase the effective constraint-to-DOF ratio and eliminate physically unrealistic conformations.

In [ ]:
labels = ['2KZV', 'GmR58A', 'HR2876B']
dofs = [184, 244, 214]
constraints = [130, 114, 97]
ratios = [d/c for d, c in zip(dofs, constraints)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(labels))
width = 0.35

ax1.bar(x - width/2, dofs, width, label='Free Parameters (DOFs)', color='#3498db')
ax1.bar(x + width/2, constraints, width, label='Experimental Constraints', color='#2ecc71')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylabel('Count')
ax1.set_title('Absolute Counts: DOFs vs Constraints')
ax1.legend()

bars2 = ax2.bar(labels, ratios, color=['#f1c40f', '#e74c3c', '#e74c3c'])
ax2.axhline(y=1.0, color='gray', linestyle='--', label='Balanced (1.0)')
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.05, f'{height:.1f}×', ha='center', va='bottom')
ax2.set_ylabel('Ratio (DOFs / Constraints)')
ax2.set_title('Imbalance Ratio')
ax2.legend()

plt.tight_layout()
plt.show()

## Comparison: diff-integrator vs. Traditional NMR Structure Refinement

| Feature | CNS/X-PLOR | CYANA | diff-integrator |
|---|---|---|---|
| **Automatic differentiation** | ❌ | ❌ | ✅ (JAX) |
| **Gradient-based optimization** | Simulated annealing | Conjugate gradient | Adam/Optax |
| **NOE distance restraints** | ✅ Thousands | ✅ Thousands | ❌ Not yet |
| **Ramachandran priors** | ✅ | ✅ | ❌ Not yet |
| **RDC restraints** | ✅ | ✅ | ✅ FixedTensorRDCLoss |
| **Chemical shift restraints** | Partial | Via CS-Rosetta | ✅ SPARTA-like |
| **Sidechain degrees of freedom** | ✅ | ✅ | ❌ Backbone only |
| **Multiple starting conformers** | ✅ Ensemble | ✅ Ensemble | ❌ Single structure |
| **Extensibility** | Limited (Fortran/TCL) | Limited | ✅ Fully differentiable |
| **Open source / PyPI** | Partial | Partial | ✅ |

### The Key Missing Ingredient: NOE Restraints

NOE (Nuclear Overhauser Effect) distance restraints are the backbone of NMR structure determination. A typical protein of 100 residues yields 1000–3000 NOE distance constraints. Without them:
1. The constraint-to-DOF ratio is far too low for reliable structure determination
2. There is no long-range structural information (chemical shifts and RDCs are local/orientation-only)
3. The global fold cannot be reliably determined from scratch

diff-integrator is currently best suited for **refinement** of an existing NMR structure model — improving local geometry and RDC agreement — not de novo structure determination.

In [ ]:
categories = ['NOE Support', 'RDC Support', 'Shift Support', 'Auto-Diff', 'Extensibility', 'Open Source']
N = len(categories)

# Scores out of 10
cns = [10, 10, 3, 0, 3, 5]
cyana = [10, 10, 7, 0, 4, 3]
diffint = [0, 8, 9, 10, 10, 10]

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
cns += cns[:1]
cyana += cyana[:1]
diffint += diffint[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

ax.plot(angles, cns, linewidth=2, linestyle='solid', label='CNS / X-PLOR')
ax.fill(angles, cns, alpha=0.1)

ax.plot(angles, cyana, linewidth=2, linestyle='solid', label='CYANA')
ax.fill(angles, cyana, alpha=0.1)

ax.plot(angles, diffint, linewidth=3, linestyle='solid', label='diff-integrator', color='#2ecc71')
ax.fill(angles, diffint, alpha=0.25, color='#2ecc71')

plt.xticks(angles[:-1], categories)
ax.set_rlabel_position(0)
plt.yticks([2, 4, 6, 8, 10], ["2", "4", "6", "8", "10"], color="grey", size=10)
plt.ylim(0, 10)

plt.title('Capability Comparison Radar', size=15, y=1.1)
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
plt.tight_layout()
plt.show()

## What Would Make diff-integrator Scientifically Stronger?

### High Priority (critical for structure determination)
1. **NOE distance restraints** — add `NOELoss(atom_pairs, upper_bounds)` term
   - Would add ~1000-3000 constraints per protein
   - Would make the system overdetermined and physically robust
   - Essential for de novo structure determination

2. **Ramachandran dihedral priors** — add `RamachandranLoss(phi, psi)` from Dunbrack database
   - Prevents non-physical backbone geometries
   - Effective DOF reducer without adding experimental cost

3. **SAXS restraints** — add small-angle X-ray scattering data
   - Provides global shape information orthogonal to NMR
   - Implemented in `diff_integrator.terms.saxs`

### Medium Priority (improvements to current approach)
4. **Cartesian coordinate parameterization with bond-length penalty** — eliminates NeRF drift for long chains
5. **Multi-conformer refinement** — optimize an ensemble of 10-20 structures jointly
6. **Full GmR58A and HR2876B RDC benchmarks** — 3 overdetermined media already available in BMRB

### Research Directions
7. **Through-SVD tensor differentiation** — carefully studied, it is currently provably incorrect for structure refinement but may have uses in tensor prediction tasks
8. **AlphaFold2 structure validation** — the Li et al. (2023) benchmark shows AF2 predictions have Q-factors comparable to NMR model 1 for many targets; diff-integrator could validate/improve AF2 predictions